# 10 — Day 6: CTU-IDSEVAL-6 Frozen External Evaluation

## Cross-Temporal Hybrid Network Intrusion Detection System

This notebook is the notebook companion to **`scripts/run_day6.py`**.

The script remains the **single source of truth** for the actual Day 6 evaluation.  
The notebook deliberately calls that script instead of re-implementing a separate version of the evaluation logic.

This prevents the notebook and script from drifting apart.

### Research rules

Day 6 is a **frozen external evaluation**:

- No Random Forest retraining
- No Isolation Forest retraining
- No threshold tuning on CTU-IDSEVAL-6
- No CTU-derived preprocessing statistics
- CIC-IDS2017 Day 1 training medians only
- Random Forest threshold = **0.01**
- Isolation Forest threshold = **0.15**
- Hybrid threshold = **0.50**
- Hybrid weights = **0.7 RF / 0.3 anomaly**

The purpose is to measure real external generalization, not to maximize CTU-IDSEVAL-6 performance.


## 1. Repository setup

The notebook may be launched from the project root or from the `notebooks/` directory.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SCRIPT = PROJECT_ROOT / "scripts" / "run_day6.py"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external" / "ctu_idseval6"
RESULTS_DIR = PROJECT_ROOT / "results" / "day6"
TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "day1" / "train.parquet"

print("Project root:", PROJECT_ROOT)
print("Day 6 script:", SCRIPT)
print("CTU-IDSEVAL-6 directory:", EXTERNAL_DIR)
print("Results directory:", RESULTS_DIR)

if not SCRIPT.exists():
    raise FileNotFoundError(f"Day 6 script not found: {SCRIPT}")


## 2. Verify frozen configuration from the script

We inspect the script text only to verify that the established frozen constants are still present.

This cell does **not** tune or derive any threshold.


In [ ]:
script_text = SCRIPT.read_text(encoding="utf-8")

expected_tokens = {
    "RF threshold": "0.01",
    "IF threshold": "0.15",
    "Hybrid threshold": "0.50",
    "RF hybrid weight": "0.7",
    "Anomaly hybrid weight": "0.3",
}

for name, token in expected_tokens.items():
    assert token in script_text, f"Expected frozen value {name}={token} not found in run_day6.py"

for forbidden in [".fit(", "GridSearchCV(", "RandomizedSearchCV("]:
    if forbidden in script_text:
        print(
            "WARNING:",
            repr(forbidden),
            "appears in script text. Inspect its context before claiming no retraining."
        )

print("Frozen-value presence check passed.")


## 3. Verify CIC-IDS2017 Flow Duration units

The Zeek adapter converts duration from seconds to CICFlowMeter-style microseconds:

`Flow Duration = Zeek duration × 1,000,000`

This conversion was checked against the actual Day 1 training distribution.

The following cell recomputes the distribution when `train.parquet` is available.


In [ ]:
if TRAIN_PATH.exists():
    train_df = pd.read_parquet(TRAIN_PATH, columns=["Flow Duration"])
    flow_duration_stats = train_df["Flow Duration"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.99]
    )
    display(flow_duration_stats)

    print("Interpretation:")
    print("- max around 120,000,000 corresponds to ~120 seconds")
    print("- 99th percentile around 118,000,000 corresponds to ~118 seconds")
    print("- This strongly supports microsecond units in this project.")
else:
    print("train.parquet is unavailable in this environment.")
    print("Previously verified real-project values:")
    print("median = 61,079")
    print("75% = 5,874,804")
    print("99% ≈ 118,038,200")
    print("max = 120,000,000")


## 4. Discover CTU-IDSEVAL-6 Zeek logs

The real Day 6 pipeline searches recursively for:

`*.conn-labeled.log`

under:

`data/external/ctu_idseval6/`

The script is responsible for parsing these Zeek logs using their `#fields` metadata.


In [ ]:
zeek_logs = sorted(
    p for p in EXTERNAL_DIR.rglob("*.conn-labeled.log")
    if p.is_file() and "_macros" not in [part.lower() for part in p.parts]
) if EXTERNAL_DIR.exists() else []

print("Zeek log files found:", len(zeek_logs))
for p in zeek_logs:
    print(" -", p.relative_to(PROJECT_ROOT))


## 5. Execute the authoritative Day 6 script

This is the central evaluation cell.

Instead of duplicating the adapter, Background policy, feature mapping, scoring, and output-writing logic, the notebook runs the real `scripts/run_day6.py`.

That guarantees the notebook uses the same:

- Zeek parser
- Zeek → CIC adapter
- label handling
- Background exclusion policy
- Day 1 training-median imputation
- Random Forest model
- Isolation Forest model
- hybrid formula
- frozen thresholds
- output schema

No notebook-specific model fitting is performed.


In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    "--external-dir",
    str(EXTERNAL_DIR),
]

print("Running:")
print(" ".join(command))

completed = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print("\n--- STDOUT ---")
print(completed.stdout)

if completed.stderr:
    print("\n--- STDERR ---")
    print(completed.stderr)

print("\nReturn code:", completed.returncode)

if completed.returncode != 0:
    raise RuntimeError(
        "Day 6 script returned a non-zero exit code. "
        "Read the output above before continuing."
    )


## 6. Load Day 6 result artifacts

The exact filenames may evolve slightly as the Day 6 implementation is refined.

This notebook loads the key result artifacts when present and reports missing optional files cleanly.


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_json(name):
    p = RESULTS_DIR / name
    if not p.exists():
        print(f"Missing optional artifact: {name}")
        return None
    return json.loads(p.read_text(encoding="utf-8"))

metrics = load_json("ctu_idseval6_metrics.json")
metadata = load_json("day6_metadata.json")
feature_mapping = load_json("feature_mapping.json")
label_mapping = load_json("label_mapping.json")

if metrics is not None:
    print("\nCTU metrics status:", metrics.get("status"))
if metadata is not None:
    print("Metadata status:", metadata.get("status"))


## 7. Dataset and Zeek parsing summary


In [ ]:
if metadata:
    parsing = metadata.get("zeek_parsing", {})
    print("Files parsed:", parsing.get("files_parsed"))
    print("Rows read:", parsing.get("rows_read"))
    print("Malformed rows:", parsing.get("malformed_rows"))

    label_dist = metadata.get("label_distribution", {})
    if label_dist:
        print("Evaluation label distribution:", label_dist)

    if "background_policy" in metadata:
        print("Background policy:", json.dumps(metadata["background_policy"], indent=2))
else:
    print("Day 6 metadata was not produced.")


### Real-project Day 6 counts

The completed real experiment produced:

- Zeek logs: **6**
- parsed rows: **286,702**
- malformed rows: **0**
- Background rows excluded: **33,163**
- rows used for binary evaluation: **253,539**
- benign: **78,742**
- malicious: **174,797**

`Background` is not automatically treated as malicious. It is unverified traffic and is therefore excluded from the binary Benign-vs-Malicious evaluation under the finalized Day 6 policy.


## 8. Feature mapping and representation compatibility

The frozen CIC-IDS2017 models expect **58 features**.

The CTU-IDSEVAL-6 source is Zeek, not CICFlowMeter, so Day 6 requires a representation adapter before the existing Day 5 feature mapper can align the external data.

The finalized experiment found a severe cross-representation mismatch:

- **9 / 58** frozen features ultimately mapped
- **49 / 58** unavailable features were filled using **CIC-IDS2017 Day 1 training medians**

This is a central Day 6 research limitation.


In [ ]:
if feature_mapping:
    print(json.dumps(feature_mapping, indent=2, default=str)[:12000])

    mapped = feature_mapping.get("mapped", {})
    unmapped = (
        feature_mapping.get("unmapped_cic2017_features")
        or feature_mapping.get("unmapped_features")
        or []
    )
    if isinstance(mapped, dict):
        print("\nMapped feature count:", len(mapped))
    if isinstance(unmapped, list):
        print("Unmapped feature count:", len(unmapped))
else:
    print("Feature mapping artifact unavailable.")
    print("Finalized real-run finding: 9/58 mapped, 49/58 training-median imputed.")


### Interpretation of the 9/58 mapping

This does **not** mean CTU-IDSEVAL-6 is a bad dataset.

It means the frozen model was trained in one feature language—CICFlowMeter—and the external dataset was recorded in another—Zeek.

Median imputation keeps the frozen evaluation reproducible and prevents leakage, but it cannot recreate information that Zeek did not provide.

Therefore the model is operating with a severely reduced representation of the external traffic.


## 9. Sanitization integrity

The mapping pipeline must ensure that no NaN or ±Inf reaches the frozen models.

All missing/unavailable values must use **CIC-IDS2017 training medians**, never statistics calculated from CTU-IDSEVAL-6.


In [ ]:
if metadata:
    sanitization = metadata.get("sanitization", {})
    print(json.dumps(sanitization, indent=2, default=str))

    checks = metadata.get("integrity_checks_passed", [])
    print("\nIntegrity checks:")
    for item in checks:
        print("✓", item)
else:
    print("Metadata not available for integrity-check display.")


## 10. Frozen configuration confirmation


In [ ]:
if metadata:
    print("Frozen thresholds:")
    print(json.dumps(metadata.get("frozen_thresholds", {}), indent=2))

    weights = (
        metadata.get("hybrid_weights_confirmed_unchanged")
        or metadata.get("hybrid_weights")
        or metadata.get("hybrid_config")
        or {}
    )
    print("\nHybrid configuration:")
    print(json.dumps(weights, indent=2, default=str))

    assert metadata.get("no_models_retrained", True) is True
    assert metadata.get("no_preprocessing_statistics_from_external_data", True) is True
    assert metadata.get("no_threshold_tuning_on_external_data", True) is True

print("\nExpected frozen values:")
print("RF threshold       = 0.01")
print("IF threshold       = 0.15")
print("Hybrid threshold   = 0.50")
print("RF weight          = 0.7")
print("Anomaly weight     = 0.3")


## 11. Detector comparison


In [ ]:
comparison_path = RESULTS_DIR / "comparison_table.csv"

if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
else:
    comparison = None

    if metrics and "comparison_table" in metrics:
        comparison = pd.DataFrame(metrics["comparison_table"])
        display(comparison)
    else:
        print("No comparison table found.")


## 12. Visual comparison of detector behavior

The most important operating-point metrics here are Recall, Precision, F1 and False Positive Rate.

A high F1 score should never be interpreted without checking the confusion matrix and FPR.


In [ ]:
if comparison is not None and not comparison.empty:
    metric_candidates = ["precision", "recall", "f1", "fpr"]
    available = [c for c in metric_candidates if c in comparison.columns]

    if available:
        plot_df = comparison.set_index("detector")[available]
        ax = plot_df.plot(kind="bar", figsize=(11, 6))
        ax.set_ylabel("Metric value")
        ax.set_title("CTU-IDSEVAL-6 — Frozen Detector Operating-Point Metrics")
        ax.set_ylim(0, 1.05)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print("Detector metrics unavailable for plotting.")


## 13. Finalized real Day 6 metrics

### Random Forest — threshold 0.01

- Precision: **0.689479**
- Recall: **0.999994**
- F1: **0.816201**
- FPR: **0.999759**
- FNR: **0.000006**
- TN: **19**
- FP: **78,723**
- FN: **1**
- TP: **174,796**
- ROC-AUC: **0.548555**
- PR-AUC: **0.726583**

The RF's F1 appears relatively high, but operationally it is extremely poor: almost every benign flow is flagged as malicious.

### Isolation Forest — threshold 0.15

- Precision: **0.095856**
- Recall: **0.001098**
- F1: **0.002172**
- FPR: **0.022999**
- FNR: **0.998902**
- TN: **76,931**
- FP: **1,811**
- FN: **174,605**
- TP: **192**
- ROC-AUC: **0.552131**
- PR-AUC: **0.711485**

The Isolation Forest almost never identifies the malicious flows at its frozen operating threshold.

### Hybrid — threshold 0.50

- Precision: **0**
- Recall: **0**
- F1: **0**
- FPR: **0**
- FNR: **1**
- TN: **78,742**
- FP: **0**
- FN: **174,797**
- TP: **0**
- ROC-AUC: **0.553548**
- PR-AUC: **0.714666**

At the frozen threshold the Hybrid predicts no malicious flows.


## 14. Attack-family results

If the Day 6 pipeline produced per-family results, they are displayed here.

These results must be interpreted under the same feature-compatibility limitation.


In [ ]:
family_path = RESULTS_DIR / "attack_family_results.csv"

if family_path.exists():
    attack_family = pd.read_csv(family_path)
    display(attack_family)
else:
    print("No attack_family_results.csv was generated.")


# Day 6 Research Interpretation

Day 6 is a **negative but scientifically valid external-generalization finding**.

The evaluation pipeline itself completed successfully using frozen models, frozen thresholds and training-only preprocessing statistics.

However, the detectors generalize poorly to CTU-IDSEVAL-6.

The strongest methodological limitation is **cross-representation feature incompatibility**:

- CIC-IDS2017 model expects 58 CICFlowMeter features.
- CTU-IDSEVAL-6 is represented by Zeek connection logs.
- Only 9/58 frozen model features were ultimately mapped.
- 49/58 were replaced with CIC-IDS2017 training medians.
- The missing information cannot be reconstructed by median imputation.

### What we can conclude

The frozen CIC-IDS2017 detectors show weak external discrimination under the available CTU-IDSEVAL-6 Zeek representation.

### What we must NOT claim

We must not claim that feature mismatch is proven to be the sole cause of the poor performance.

Other forms of dataset shift may also contribute.

### Why we do not tune Day 6

Changing thresholds, retraining models, selecting CTU-specific features or learning preprocessing from CTU-IDSEVAL-6 would change the research question.

The proposal asks whether the **frozen system generalizes** to a new 2026 dataset.

Therefore the negative result is retained and reported honestly.


## 15. Reproducibility checklist


In [ ]:
checklist = pd.DataFrame([
    ["Random Forest retrained?", "No"],
    ["Isolation Forest retrained?", "No"],
    ["CTU preprocessing statistics used?", "No"],
    ["Threshold tuned on CTU?", "No"],
    ["RF threshold", "0.01"],
    ["IF threshold", "0.15"],
    ["Hybrid threshold", "0.50"],
    ["RF hybrid weight", "0.7"],
    ["Anomaly hybrid weight", "0.3"],
    ["Background included as attack?", "No — excluded from binary metrics"],
    ["Frozen expected feature count", "58"],
    ["Ultimately mapped CTU features", "9"],
    ["Training-median-imputed features", "49"],
    ["Duration conversion", "Zeek seconds × 1,000,000 → microseconds"],
], columns=["Check", "Day 6 result"])

display(checklist)


## 16. Next research stage

After freezing and documenting Day 6, the project proceeds to **Day 7 — Explainability**:

- Random Forest TreeSHAP / SHAP
- local TP/TN/FP/FN explanations
- permutation-importance cross-check
- Isolation Forest explainability
- structured evidence for the later LLM explanation layer

The Day 6 limitation must remain visible during explainability: imputed CTU features must not be presented as genuinely observed measurements.
